# C3: contiguous KV cache

**Goal:** prefill once, then process only the new token. Reuse old K/V at every layer.

**Spec:** cached FP32 logits must match our no-cache model and HF within B8 tolerances. Greedy tokens must match exactly. C2's EOS, sampling, context, dtype/device and immutability contracts remain unchanged. Zero-token requests do not allocate a cache.

Run top to bottom on a **fresh Colab T4**. Two FP32 model copies need about 8.8 GB of weight memory. If loading kills the kernel, replace the whole runtime. This lesson checks correctness; **C4 will measure speed**.

Runtime validation is pending your run. All implementation lives in the repository, not notebook writefile cells.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


In [ ]:
import gc
import torch
from tinyserve.generation import generate
from tinyserve.models.attention import AttentionMetadata
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.probe import compare_cached_steps
from tinyserve.basics import load_model, encode
from tinyserve.envinfo import collect_env, save_result
from tinyserve.weights import kv_cache_bytes_per_token

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


## 1. Test first

Run B8/C1/C2/C3 tests. C3 covers prefill and decode chunks, MHA/GQA/MQA, exact memory cost, post-RoPE keys, unchanged old entries, reset, overflow, position errors, failure/retry, greedy parity and seeded replay.

B8 and C2 regression tests must still pass. One two-GPU test can skip on a single T4. CUDA FP16 tests should run here. A failed test stops the notebook.

In [ ]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b8_model.py", "tests/test_c1_sampling.py",
                           "tests/test_c2_generation.py", "tests/test_c3_cache.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. Why caching is valid, and what it stores

Causal attention prevents old tokens from seeing new tokens. Their hidden states and K/V stay unchanged when we append a token, for fixed weights and positions.

Each storage tensor is `(22 layers, capacity tokens, 4 KV heads, 64 dimensions)`. We cache **four KV heads**, not the 32 expanded query heads. Keys are stored **after RoPE**; values are not rotated. Queries are not cached.

`2 * 22 * 4 * 64 * 2 = 22,528 bytes = 22 KiB/token` in FP16. FP32 costs 44 KiB/token. A 2,048-token reservation costs 44 MiB in FP16 or 88 MiB in FP32, excluding weights and temporaries.

Load the same FP32 weights into HF and our model. HF is the independent reference, not our cached forward implementation.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
raw_ids = encode(tokenizer, "The capital of France is", DEVICE)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France? Answer in one sentence."},
]
chat_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True)["input_ids"][0].to(DEVICE)
print("raw tokens:", raw_ids.numel(), "chat tokens:", chat_ids.numel())
print("FP16 KiB/token:", kv_cache_bytes_per_token(our_model.config, 2) / 1024)
print("FP32 KiB/token:", kv_cache_bytes_per_token(our_model.config, 4) / 1024)


## 3. Watch prefill become one-token decode

For a prompt of length P: prefill uses positions `0..P-1`, then decode uses P, P+1, ... . Do not restart at zero or rotate old keys again.

The verification helper checks cached logits against our full-prefix logits and HF at **every step**. Prefill checks all prompt rows; decode checks its new row. FP32 uses `rtol=1e-5, atol=1e-4`, as in B8. Token equality is a separate assertion because close scores can swap argmax.

Inspect `forward_tokens`: only the first row should contain the full prompt. Cache length advances after all 22 layers succeed. The last generated token is returned without another forward, so cache length ends one token short of returned length.

In [ ]:
parity_rows = []
for label, prompt_ids, budget in (("raw", raw_ids, 12), ("chat", chat_ids, 24)):
    generated_ids, step_rows = compare_cached_steps(our_model, hf_model, prompt_ids, max_new_tokens=budget)
    direct_ids = generate(our_model, prompt_ids, max_new_tokens=budget, use_cache=True)
    assert torch.equal(generated_ids, direct_ids), f"public cached generator mismatch for {label}"
    with torch.no_grad():
        hf_ids = hf_model.generate(prompt_ids[None], attention_mask=torch.ones_like(prompt_ids)[None],
                                   do_sample=False, max_new_tokens=budget, use_cache=True,
                                   pad_token_id=hf_model.config.eos_token_id)[0]
    assert torch.equal(generated_ids, hf_ids), f"HF cached greedy token parity failed for {label}"
    continuation = generated_ids[prompt_ids.numel():]
    answer = tokenizer.decode(continuation.tolist(), skip_special_tokens=True)
    for row in step_rows:
        print(f"{label} step={row['step']:2d} forward_tokens={row['forward_tokens']:3d} "
              f"cache_length={row['cache_length']:3d} "
              f"diff(no-cache)={row['cached_vs_no_cache_max_abs_diff']:.2e} "
              f"diff(HF)={row['cached_vs_hf_max_abs_diff']:.2e}")
    parity_rows.append({"prompt_type": label, "prompt_tokens": prompt_ids.numel(), "budget": budget,
                        "token_ids": generated_ids.tolist(), "continuation": answer,
                        "new_tokens": continuation.numel(), "hf_cached_token_equal": True, "steps": step_rows})
    print(label, "answer:", repr(answer))


## 4. Read the rectangular causal mask

Free HF now; only our model is needed below. Inspect a two-token chunk after the raw prompt. The first new query sees all old keys and itself, but not the second new token. The second query sees all keys.

This is **not** `tril(2, P+2)`. Visibility uses absolute positions: `key_position <= query_position`. Capacity is reserved storage; only the committed prefix and current chunk are visible.

In [ ]:
del hf_model
gc.collect()
torch.cuda.empty_cache()
demo_cache = our_model.new_kv_cache(raw_ids.numel() + 2)
with torch.no_grad():
    prefill_positions = torch.arange(raw_ids.numel(), device=raw_ids.device)
    our_model(raw_ids, prefill_positions, AttentionMetadata(prefill_positions, cache=demo_cache))
chunk_positions = torch.arange(demo_cache.length, demo_cache.length + 2, device=raw_ids.device)
chunk_mask = AttentionMetadata(chunk_positions, cache=demo_cache).allowed_mask()
assert bool(chunk_mask[0, :-1].all()) and not bool(chunk_mask[0, -1]) and bool(chunk_mask[1].all())
print("query positions:", chunk_positions.tolist())
print("visible keys (1 means visible):\n", chunk_mask.int().cpu())
print("K/V shape:", tuple(demo_cache.key.shape), "committed length:", demo_cache.length)
print("reserved bytes:", demo_cache.allocated_bytes, "bytes/token:", demo_cache.bytes_per_token)
memory_row = {"shape_per_tensor": list(demo_cache.key.shape), "capacity": demo_cache.capacity,
              "committed_length": demo_cache.length, "allocated_bytes": demo_cache.allocated_bytes,
              "bytes_per_token": demo_cache.bytes_per_token, "chunk_mask": chunk_mask.int().tolist()}


## 5. Boundaries and reset

Cached positions must append from `cache.length`. Overflow and packed request IDs raise. A cache belongs to one model and one request at a time; recreate it after moving/casting, and reset or recreate it after changing weights.

`reset()` forgets the prefix without reallocating. Old bytes remain but are hidden by length. Direct cached forwards require eval mode and `torch.no_grad()`. The public generator handles both no-grad and fresh allocation.

Verify the same storage can serve a new prompt without leaking the old request. Then check C2's zero-budget and strict context behavior.

In [ ]:
try:
    with torch.no_grad():
        wrong_positions = torch.zeros(1, device=raw_ids.device, dtype=torch.int64)
        our_model(raw_ids[-1:], wrong_positions, AttentionMetadata(wrong_positions, cache=demo_cache))
except ValueError as error:
    position_error = str(error)
    print("expected position error:", position_error)
else:
    raise AssertionError("decode positions must not restart at zero")
storage_pointer = demo_cache.key.data_ptr()
demo_cache.reset()
assert demo_cache.length == 0 and demo_cache.key.data_ptr() == storage_pointer
with torch.no_grad():
    short_ids = raw_ids[:2]
    short_positions = torch.arange(short_ids.numel(), device=short_ids.device)
    reset_logits = our_model(short_ids, short_positions, AttentionMetadata(short_positions, cache=demo_cache))
    plain_logits = our_model(short_ids, short_positions, AttentionMetadata(short_positions))
torch.testing.assert_close(reset_logits, plain_logits, rtol=1e-5, atol=1e-4)
zero_ids = generate(our_model, raw_ids, max_new_tokens=0, use_cache=True)
assert torch.equal(zero_ids, raw_ids) and zero_ids.data_ptr() != raw_ids.data_ptr()
try:
    generate(our_model, raw_ids, max_new_tokens=our_model.config.max_position_embeddings, use_cache=True)
except ValueError as error:
    context_error = str(error)
    print("expected context error:", context_error)
else:
    raise AssertionError("oversized request should have raised")
reset_max_abs_diff = (reset_logits - plain_logits).abs().max().item()


## 6. Cached sampling and replay

C1 sampling is unchanged. Recreate seed 17 to verify repeatability on this runtime. No sampled token equality between different implementations or devices is claimed.

Each `generate` call creates its own cache, so the second request cannot inherit the first request's K/V.

In [ ]:
sample_options = dict(max_new_tokens=24, temperature=0.8, top_k=40, top_p=0.9, use_cache=True)
sampled_ids = generate(our_model, chat_ids,
                       generator=torch.Generator(device=chat_ids.device).manual_seed(17), **sample_options)
repeated_ids = generate(our_model, chat_ids,
                        generator=torch.Generator(device=chat_ids.device).manual_seed(17), **sample_options)
assert torch.equal(sampled_ids, repeated_ids)
sampled_answer = tokenizer.decode(sampled_ids[chat_ids.numel():].tolist(), skip_special_tokens=True)
print("sampled answer:", repr(sampled_answer))
print("same-seed replay equal:", torch.equal(sampled_ids, repeated_ids))


## 7. Save evidence and three interview questions

1. **Why can earlier K/V be reused?** Causality makes old hidden states independent of appended tokens, for fixed weights and positions.
2. **Why 22 KiB/token in FP16?** Both K and V, all 22 layers, four KV heads, 64 dimensions, two bytes per element.
3. **Does cache make decode constant-cost?** No. Each query still reads the growing K/V prefix, and every step reads model weights.

Save the JSON and notebook **with outputs** before replacing your runtime. The ranges below describe numerical differences, not timing trials. **Next: C4, measured no-cache versus cached speed curve.**

In [ ]:
all_steps = [step for row in parity_rows for step in row["steps"]]
no_cache_diffs = [step["cached_vs_no_cache_max_abs_diff"] for step in all_steps]
hf_diffs = [step["cached_vs_hf_max_abs_diff"] for step in all_steps]
saved_path = save_result("C3_kv_cache", {
    "dtype": "float32", "use_cache": True, "single_request": True,
    "greedy_comparison": parity_rows, "cache_memory": memory_row,
    "cached_vs_no_cache_diff_range": {"min": min(no_cache_diffs), "max": max(no_cache_diffs)},
    "cached_vs_hf_diff_range": {"min": min(hf_diffs), "max": max(hf_diffs)},
    "position_error": position_error, "oversized_context_error": context_error,
    "reset_max_abs_diff": reset_max_abs_diff, "reset_reused_storage": True,
    "zero_budget_returns_prompt": torch.equal(zero_ids, raw_ids),
    "sampled_generation": {"temperature": 0.8, "top_k": 40, "top_p": 0.9, "seed": 17,
                           "token_ids": sampled_ids.tolist(), "continuation": sampled_answer,
                           "same_seed_replay_equal": torch.equal(sampled_ids, repeated_ids)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
del demo_cache, our_model, reset_logits, plain_logits
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
